# LLM-as-judge aggregate analysis

Analysis companion to `llm_judge.md`. Computes the per-strategy distributions, surface-vs-judge correlations, and disagreement region counts that the writeup references, and generates the figures committed to `evaluation/results/llm_judge/`.

**Data input:** `evaluation/runs/llm_judge_aggregate_v1.csv` — 16,962 variants from the Day 2 scoring run, of which 16,959 were successfully scored by the judge (99.98% success rate).

**Outputs:**
- `per_strategy_distributions.png` — boxplots of judge Likert scores by strategy, both dimensions
- `sim_vs_equiv_scatter.png` — surface similarity vs judge equivalence, per strategy
- `ppl_vs_natural_scatter.png` — log perplexity vs judge naturalness, per strategy
- `disagreement_regions.png` — visualization of where variants cluster relative to the high-sim/low-equiv disagreement quadrant
- `judge_aggregate_summary.csv` — per-strategy statistics table

## Setup

In [ ]:
import csv
import math
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# ─── Configuration ──────────────────────────────────────────
AGGREGATE = Path("../runs/llm_judge_aggregate_v1.csv")
RESULTS_DIR = Path("../results/llm_judge")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Strategy order and colors match the convention in scoring_ranges_analysis
# and prompt_revisions_analysis so cross-notebook figures look consistent.
STRATEGY_ORDER = ["llm_paraphrase", "constrained", "mlm_substitution", "expansion"]
STRATEGY_COLORS = {
    "llm_paraphrase": "#2E86AB",   # blue
    "constrained": "#A23B72",      # magenta
    "mlm_substitution": "#E07A5F", # orange
    "expansion": "#81B29A",        # green
}


In [ ]:
def load_aggregate(path):
    """Load the Day 2 aggregate, keeping only successfully-scored rows with
    valid numeric scores. Convert score and surface metric columns to numeric."""
    rows = []
    status_counts = Counter()
    with open(path) as f:
        for r in csv.DictReader(f):
            status_counts[r["judge_status"]] += 1
            if r["judge_status"] != "scored":
                continue
            try:
                r["judge_equiv"] = int(r["judge_semantic_equivalence"])
                r["judge_natural"] = int(r["judge_naturalness"])
                r["sim"] = float(r["semantic_similarity"])
                r["ppl"] = float(r["perplexity"])
                if r["ppl"] <= 0:
                    continue
                r["log_ppl"] = math.log(r["ppl"])
            except (ValueError, KeyError):
                continue
            rows.append(r)
    return rows, status_counts

rows, status_counts = load_aggregate(AGGREGATE)
print(f"Loaded {len(rows):,} successfully-scored variants")
print(f"Status breakdown: {dict(status_counts)}")

# Bucket by strategy_family for per-strategy analysis
by_strategy = defaultdict(list)
for r in rows:
    by_strategy[r["strategy_family"]].append(r)

print()
print("Variants per strategy:")
for s in STRATEGY_ORDER:
    print(f"  {s}: {len(by_strategy[s]):,}")


---

## Section 1: Per-strategy summary

Headline statistics on both judge dimensions, with failure rates (% ≤2),
high rates (% ≥4), and clean rates (≥4 on both dimensions).

In [ ]:
def summarize(strategy_rows, key):
    vals = [r[key] for r in strategy_rows]
    n = len(vals)
    return {
        "n": n,
        "mean": round(sum(vals) / n, 2),
        "median": sorted(vals)[n // 2],
        "pct_fail": round(sum(1 for v in vals if v <= 2) / n * 100, 1),
        "pct_high": round(sum(1 for v in vals if v >= 4) / n * 100, 1),
    }

summary_rows = []
for strat in STRATEGY_ORDER:
    s_rows = by_strategy[strat]
    eq = summarize(s_rows, "judge_equiv")
    nat = summarize(s_rows, "judge_natural")
    clean = round(
        sum(1 for r in s_rows if r["judge_equiv"] >= 4 and r["judge_natural"] >= 4)
        / len(s_rows) * 100, 1
    )
    # Median surface metrics for context
    sims = sorted(r["sim"] for r in s_rows)
    ppls = sorted(r["ppl"] for r in s_rows)
    summary_rows.append({
        "strategy": strat,
        "n": eq["n"],
        "eq_mean": eq["mean"],
        "eq_median": eq["median"],
        "eq_%≤2": eq["pct_fail"],
        "eq_%≥4": eq["pct_high"],
        "nat_mean": nat["mean"],
        "nat_median": nat["median"],
        "nat_%≤2": nat["pct_fail"],
        "nat_%≥4": nat["pct_high"],
        "clean_%": clean,
        "sim_median": round(sims[len(sims) // 2], 2),
        "ppl_median": round(ppls[len(ppls) // 2], 1),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_DIR / "judge_aggregate_summary.csv", index=False)
summary_df


The two columns to read first are `clean_%` (full quality success rate) and
the gap between `sim_median` and `eq_median`:

- `llm_paraphrase` has a clean rate of 88.8% and surface-vs-judge alignment
  (sim 0.85, eq 5).
- `constrained` is the workhorse at 54.9% clean (per-transform breakdown in
  `prompt_revisions.md` explains the variation).
- `mlm_substitution` has the central finding: median sim 0.87 (high) but
  median eq 2 (low). The surface metric and judge measure different things
  on this strategy.
- `expansion` is dimension-asymmetric: median nat 5 but median eq 2. This
  is by design — expansion intentionally generates adjacent intents, not
  paraphrases.

---

## Section 2: Per-strategy distributions

Boxplots showing the full Likert distribution per strategy on both judge
dimensions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
labels = ["semantic equivalence", "naturalness"]
keys = ["judge_equiv", "judge_natural"]

for ax, key, label in zip(axes, keys, labels):
    data = [
        [r[key] for r in by_strategy[s]]
        for s in STRATEGY_ORDER
    ]
    bp = ax.boxplot(
        data,
        labels=STRATEGY_ORDER,
        patch_artist=True,
        widths=0.6,
        showmeans=True,
        meanprops={"marker": "D", "markerfacecolor": "white",
                   "markeredgecolor": "black", "markersize": 7},
        medianprops={"color": "black", "linewidth": 1.5},
        flierprops={"marker": ".", "markersize": 4, "alpha": 0.4},
    )
    # Color each box by strategy
    for patch, strat in zip(bp["boxes"], STRATEGY_ORDER):
        patch.set_facecolor(STRATEGY_COLORS[strat])
        patch.set_alpha(0.7)

    ax.set_title(f"Judge {label} by strategy", fontsize=12)
    ax.set_ylabel("Likert score" if key == "judge_equiv" else "")
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_ylim(0.5, 5.5)
    ax.grid(True, alpha=0.3, axis="y")

    # Rotate x labels
    for tick in ax.get_xticklabels():
        tick.set_rotation(15)
        tick.set_ha("right")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "per_strategy_distributions.png", dpi=120, bbox_inches="tight")
plt.show()


The boxplots make the four strategies' distinct profiles visible at a glance:

- `llm_paraphrase` clusters tight against the 4-5 ceiling on both dimensions.
- `constrained` has wider distributions reflecting per-transform variation.
- `mlm_substitution` has the lowest equivalence median (2) and a wide naturalness IQR — mechanical substitutions are often grammatically locally-natural but semantically off.
- `expansion` has a median of 5 on naturalness with very tight distribution (LLM-generated utterances are fluent by construction) and a wide equivalence distribution centered at 2-3 (low by design, since expansion generates adjacent intents rather than paraphrases).

---

## Section 3: Surface-metric vs judge correlations

Spearman rank correlation between the surface metrics (similarity,
perplexity) and the corresponding judge dimensions (equivalence,
naturalness), overall and per strategy.

In [ ]:
# Overall correlations
sims_all = [r["sim"] for r in rows]
eqs_all = [r["judge_equiv"] for r in rows]
log_ppls_all = [r["log_ppl"] for r in rows]
nats_all = [r["judge_natural"] for r in rows]

rho_sim_eq, _ = spearmanr(sims_all, eqs_all)
rho_lppl_nat, _ = spearmanr(log_ppls_all, nats_all)

print(f"Overall Spearman correlations (n={len(rows):,}):")
print(f"  similarity ↔ judge_equiv:      {rho_sim_eq:+.3f}")
print(f"  log(perplexity) ↔ judge_natural: {rho_lppl_nat:+.3f}")
print()

# Per-strategy correlations
print(f"{'strategy':<22s} {'sim↔eq':>10s} {'log(ppl)↔nat':>16s}")
print("-" * 50)
corr_rows = []
for strat in STRATEGY_ORDER:
    s_rows = by_strategy[strat]
    sims_s = [r["sim"] for r in s_rows]
    eqs_s = [r["judge_equiv"] for r in s_rows]
    lps_s = [r["log_ppl"] for r in s_rows]
    nts_s = [r["judge_natural"] for r in s_rows]
    rse, _ = spearmanr(sims_s, eqs_s)
    rpn, _ = spearmanr(lps_s, nts_s)
    print(f"{strat:<22s} {rse:>+10.3f} {rpn:>+16.3f}")
    corr_rows.append({
        "strategy": strat,
        "rho_sim_equiv": round(rse, 3),
        "rho_logppl_nat": round(rpn, 3),
    })

corr_df = pd.DataFrame(corr_rows)


The overall correlations are small (+0.224, −0.121). Per strategy, MLM has
the strongest sim↔eq correlation (+0.404) — but this reflects within-strategy
variation rather than agreement on average level. MLM's similarity
distribution is shifted high (median 0.87) while its equivalence distribution
is shifted low (median 2), so most MLM variants live in the surface
"looks good" region while being judged "doesn't match".

In [ ]:
def scatter_per_strategy(x_key, y_key, x_label, y_label, save_name, title,
                          y_ticks=[1, 2, 3, 4, 5], y_lim=(0.5, 5.5)):
    """4-panel scatter: one panel per strategy, points alpha-blended,
    Spearman correlation annotated."""
    fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
    for ax, strat in zip(axes, STRATEGY_ORDER):
        s_rows = by_strategy[strat]
        xs = [r[x_key] for r in s_rows]
        ys = [r[y_key] for r in s_rows]
        # Jitter y-axis for the Likert dimension so points don't all stack
        ys_jit = [y + np.random.uniform(-0.2, 0.2) for y in ys]

        ax.scatter(xs, ys_jit, color=STRATEGY_COLORS[strat],
                   alpha=0.12, s=8, edgecolors="none")

        rho, _ = spearmanr(xs, ys)
        ax.text(0.05, 0.95, f"ρ = {rho:+.3f}",
                transform=ax.transAxes, fontsize=11, verticalalignment="top",
                bbox={"boxstyle": "round,pad=0.3", "facecolor": "white",
                      "edgecolor": "gray", "alpha": 0.8})

        ax.set_title(f"{strat}\n(n={len(s_rows):,})", fontsize=11)
        ax.set_xlabel(x_label, fontsize=10)
        ax.set_yticks(y_ticks)
        ax.set_ylim(*y_lim)
        ax.grid(True, alpha=0.3)

    axes[0].set_ylabel(y_label, fontsize=11)
    fig.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / save_name, dpi=120, bbox_inches="tight")
    plt.show()

scatter_per_strategy(
    x_key="sim", y_key="judge_equiv",
    x_label="Semantic similarity (all-MiniLM-L6-v2 cosine)",
    y_label="Judge semantic equivalence (jittered)",
    save_name="sim_vs_equiv_scatter.png",
    title="Surface similarity vs judge equivalence, per strategy",
)


In [ ]:
scatter_per_strategy(
    x_key="log_ppl", y_key="judge_natural",
    x_label="Log perplexity (GPT-2)",
    y_label="Judge naturalness (jittered)",
    save_name="ppl_vs_natural_scatter.png",
    title="Log perplexity vs judge naturalness, per strategy",
)


The scatters show what the correlation numbers compress:

- For `llm_paraphrase`, both metrics cluster in the upper-right (high
  surface, high judge). Variants there are uniformly good and the surface
  metrics correctly flag them.
- For `mlm_substitution`, the similarity scatter shows a clear horizontal
  band of variants with high similarity (>0.85) and low judge equivalence
  (1-2). This is the disagreement region we quantify in the next section.
- For `expansion`, the equivalence scatter shows a broad cloud with no
  clear gradient — similarity does not predict whether the judge will
  rate a variant equivalent.

---

## Section 4: Disagreement regions

The most action-relevant view of judge-surface divergence: counts of
variants where the two metrics sharply disagree. These are the cases where
a practitioner relying on surface scores alone would draw a different
conclusion from what the judge reports.

In [ ]:
# Region A: high similarity, low equivalence
high_sim_low_eq = [r for r in rows if r["judge_equiv"] <= 2 and r["sim"] > 0.85]
print(f"Variants with judge_equiv ≤ 2 AND similarity > 0.85: {len(high_sim_low_eq):,}")
breakdown_a = Counter(r["strategy_family"] for r in high_sim_low_eq)
for strat, n in breakdown_a.most_common():
    pct = n / len(high_sim_low_eq) * 100
    print(f"  {strat}: {n:,} ({pct:.1f}%)")

print()

# Region B: low perplexity, low naturalness
low_ppl_low_nat = [r for r in rows if r["judge_natural"] <= 2 and r["ppl"] < 200]
print(f"Variants with judge_natural ≤ 2 AND perplexity < 200: {len(low_ppl_low_nat):,}")
breakdown_b = Counter(r["strategy_family"] for r in low_ppl_low_nat)
for strat, n in breakdown_b.most_common():
    pct = n / len(low_ppl_low_nat) * 100
    print(f"  {strat}: {n:,} ({pct:.1f}%)")


In [ ]:
# Visualization: 4-panel scatter, each strategy, with the disagreement
# quadrant shaded. Mirrors the per-strategy scatter from Section 3 but
# zooms in on the disagreement region's prominence per strategy.

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharey=True)
for ax, strat in zip(axes, STRATEGY_ORDER):
    s_rows = by_strategy[strat]
    xs = [r["sim"] for r in s_rows]
    ys = [r["judge_equiv"] + np.random.uniform(-0.2, 0.2) for r in s_rows]

    # Shade the disagreement quadrant
    ax.axhspan(0.5, 2.5, xmin=(0.85 - 0)/1.0, xmax=1.0,
               color="red", alpha=0.08, zorder=0)
    ax.axhline(2.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.axvline(0.85, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)

    ax.scatter(xs, ys, color=STRATEGY_COLORS[strat],
               alpha=0.12, s=8, edgecolors="none", zorder=1)

    # Count variants in the disagreement region for this strategy
    n_disagree = sum(1 for r in s_rows
                     if r["judge_equiv"] <= 2 and r["sim"] > 0.85)
    pct_disagree = n_disagree / len(s_rows) * 100
    ax.text(0.05, 0.95,
            f"in disagreement region:\n{n_disagree:,} ({pct_disagree:.1f}%)",
            transform=ax.transAxes, fontsize=10, verticalalignment="top",
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "white",
                  "edgecolor": "gray", "alpha": 0.85})

    ax.set_title(f"{strat}\n(n={len(s_rows):,})", fontsize=11)
    ax.set_xlabel("Semantic similarity", fontsize=10)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_ylim(0.5, 5.5)
    ax.set_xlim(0, 1.0)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Judge semantic equivalence (jittered)", fontsize=11)
fig.suptitle(
    "Disagreement region: variants with similarity > 0.85 but judge_equiv ≤ 2\n"
    "(shaded quadrant = surface metric says 'good', judge says 'wrong')",
    fontsize=12, y=1.03
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "disagreement_regions.png", dpi=120, bbox_inches="tight")
plt.show()


The shaded quadrant is the disagreement region: surface similarity > 0.85
and judge equivalence ≤ 2. The proportion of each strategy's variants
falling in this region is the practitioner-relevant headline:

- `mlm_substitution`: ~35% of MLM variants live in the disagreement region.
  This is the central finding — if a practitioner gates ingestion on
  similarity > 0.85, more than one in three MLM variants will be false
  positives by judge measurement.
- `constrained`: 5.6% in disagreement region. Substantial but lower.
- `expansion` and `llm_paraphrase`: under 2% each.

The structural takeaway is that surface similarity is a reasonable filter
for the LLM-generated strategies (`llm_paraphrase`, `expansion`, even
`constrained`) but is misleading for MLM substitution. The
`scoring_ranges.md` writeup already noted that surface similarity has
known blind spots; the judge data quantifies the cost of those blind spots
when surface scores are used as a quality gate.

---

## Summary

1. **Per-strategy clean rates span an order of magnitude:** `llm_paraphrase`
   88.8%, `constrained` 54.9%, `expansion` 14.4%, `mlm_substitution` 8.8%.
   Each strategy fails differently — MLM on equivalence, expansion on
   equivalence (by design), constrained on a subset of transforms
   (addressed in `prompt_revisions.md`).

2. **Surface-vs-judge correlations are weak overall** (+0.224 sim↔eq,
   −0.121 log(ppl)↔nat) and vary substantially by strategy. The two
   metric families measure different things.

3. **The disagreement region is dominated by MLM:** 1,840 of 2,178
   variants with sim > 0.85 but judge_equiv ≤ 2 are MLM substitutions
   (84.5%). For MLM specifically, ~35% of all variants fall in the
   disagreement region.

4. **Practitioner implication:** Surface similarity is a defensible
   quality gate for the LLM-driven strategies but should not be relied
   on as the sole gate for MLM substitution output. Either inspect MLM
   variants manually before ingestion, or filter on a third quality signal
   (e.g., judge scoring behind a `--use-judge` flag — see future work in
   `llm_judge.md`).

## Files generated

All five outputs land in `evaluation/results/llm_judge/`:

- `per_strategy_distributions.png`
- `sim_vs_equiv_scatter.png`
- `ppl_vs_natural_scatter.png`
- `disagreement_regions.png`
- `judge_aggregate_summary.csv`